In [ ]:
from pathlib import Path
import pandas as pd
from typing import Literal

In [ ]:
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
CHUNK_SIZE = 500_000
DATASET = {
    "payments": "General_Payments_2024.csv",
    "prescribers": "Med_Prescribers_2024.csv",
}
CLEANED_SET ={
    "company": "cleaned_companies.csv",
    "doctor": "cleaned_doctors.csv",
    "prescription": "cleaned_prescriptions.csv",
    "drug": "cleaned_drugs.csv",
    "payment": "cleaned_payments.csv",
    "payment_by_drug": "cleaned_payment_by_drug.csv"
}

In [ ]:
# Load file into DataFrames using a chunk size to limit the number of rows read
def load_in_chunks(file_path, chunk_size, file_type: Literal["json", "csv", "xlsx", "xls"]="csv") -> pd.DataFrame:
    try:
        chunks = []
        chunk_no = 0
        if file_type == "json":
            for chunk in pd.read_json(file_path, lines=True, chunksize=chunk_size):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        elif file_type == "csv":
            for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        elif file_type in ["xlsx", "xls"]:
            for chunk in pd.read_excel(file_path, chunksize=chunk_size):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        return pd.concat(chunks, ignore_index=True)
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        return pd.DataFrame()

def get_file_extension(file_path: str) -> str:
    try:
        return Path(file_path).suffix.lower().lstrip(".")
    except Exception as e:
        print(f"Error getting file extension for {file_path}: {e}")
        return ""


In [ ]:
payments_path = DATA_DIR / DATASET["payments"]
prescribers_path = DATA_DIR / DATASET["prescribers"]

payments_df = load_in_chunks(payments_path, CHUNK_SIZE, file_type=get_file_extension(payments_path))
prescribers_df = load_in_chunks(prescribers_path, CHUNK_SIZE, file_type=get_file_extension(prescribers_path))

In [ ]:
# Print the shape and first few rows of the payments dataframe
print(payments_df.shape)
payments_df.head(3)

In [ ]:
# Print the shape and first few rows of the prescribers dataframe
print(prescribers_df.shape)
prescribers_df.head(3)

In [ ]:
# Check for duplicated rows in the prescribers dataframe based on NPI, Brand Name, and Generic Name
prescribers_df.duplicated(
    ["Prscrbr_NPI", "Brnd_Name", "Gnrc_Name"]
).sum()

In [ ]:
key_columns = ["Prscrbr_NPI", "Brnd_Name", "Gnrc_Name"]

# Missing values in each candidate key column
print(prescribers_df[key_columns].isna().sum())

# Total rows with at least one missing key value
print(
    "Rows with a missing key value:",
    prescribers_df[key_columns].isna().any(axis=1).sum()
)

In [ ]:
print(payments_df.info())
print(prescribers_df.info())

In [ ]:
# Drop rows with missing "Covered_Recipient_NPI" values and ensure the column is of integer type for consistency with prescribers_df.
print("Covered_Recipient_NPI Data Type:" ,payments_df["Covered_Recipient_NPI"].dtype)
print("Prscrbr_NPI Data Type:" ,prescribers_df["Prscrbr_NPI"].dtype)
print("Count of missing Covered_Recipient_NPI:", payments_df["Covered_Recipient_NPI"].isna().sum())

payments_df.dropna(subset=["Covered_Recipient_NPI"], inplace=True)
payments_df["Covered_Recipient_NPI"] = payments_df["Covered_Recipient_NPI"].astype("int64")


In [ ]:
# Collect Company Information
company_df = (
    payments_df[[
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
        # "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State",
        # "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country",
    ]]
    .drop_duplicates(subset=["Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID"])
    .reset_index(drop=True)
).rename(columns={
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID": "company_id",
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name": "company_name",
    # "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State": "state",
    # "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country": "country",
})


# Ensure that each company has a unique Company_ID in the final company dimension.
assert company_df["company_id"].is_unique
company_df.head()

##### DOCTOR RESOLUTION

In [ ]:
# Check for NPIs prescribing to multiple states. If none exists, then the data is consistent with single-state prescribing.
# This is to validate that the state in Medicare Data corresponds to the NPIs actual state of practice
states_per_npi = prescribers_df.groupby("Prscrbr_NPI")["Prscrbr_State_Abrvtn"].nunique()
multi_state = (states_per_npi > 1).sum()

print("NPIs prescribing to multiple states:", multi_state)
print("Check passed" if multi_state == 0 else "Check failed")

In [ ]:
# Collect Prescriber/Doctor information and resolve specialty from payment data. Inner join used to avoid unmatched doctors. i.e Only consider doctors with payment records
pres = prescribers_df[[
    "Prscrbr_NPI",
    "Prscrbr_Last_Org_Name",
    "Prscrbr_First_Name",
    "Prscrbr_State_Abrvtn"
]].drop_duplicates(subset=["Prscrbr_NPI"])

pay = payments_df[[
    "Covered_Recipient_NPI",
    "Recipient_City",
    "Covered_Recipient_Specialty_1",
    "Covered_Recipient_Primary_Type_1",
    "Recipient_Primary_Business_Street_Address_Line1",
    "Recipient_Primary_Business_Street_Address_Line2"
]].drop_duplicates(subset=["Covered_Recipient_NPI"])


doctor_df = pres.merge(
    pay,
    left_on="Prscrbr_NPI",
    right_on="Covered_Recipient_NPI",
    how="inner"
).reset_index(drop=True)


doctor_df = doctor_df[[
    "Prscrbr_NPI",
    "Prscrbr_Last_Org_Name",
    "Prscrbr_First_Name",
    "Covered_Recipient_Specialty_1",
    "Covered_Recipient_Primary_Type_1",
    "Prscrbr_State_Abrvtn",
    "Recipient_City",
    "Recipient_Primary_Business_Street_Address_Line1",
    "Recipient_Primary_Business_Street_Address_Line2"
]].rename(columns={
    "Prscrbr_NPI": "npi",
    "Prscrbr_Last_Org_Name": "last_name",
    "Prscrbr_First_Name": "first_name",
    "Covered_Recipient_Specialty_1": "specialty",
    "Prscrbr_State_Abrvtn": "state",
    "Recipient_City": "city",
    "Covered_Recipient_Primary_Type_1": "primary_type",
    "Recipient_Primary_Business_Street_Address_Line1": "address_line1",
    "Recipient_Primary_Business_Street_Address_Line2": "address_line2"
})

# Ensure that each doctor has a unique NPI in the final doctor dimension.
assert doctor_df["npi"].is_unique
assert doctor_df["state"].notnull().all()

print("Doctors Row Count:", doctor_df.shape[0])
doctor_df.head()



##### DRUG RESOLUTION
Because Brand Name maps to more than one Generic Name, it will be impossible to use a brand name as the primary key. 

Will use a surrogate key "drug_id" as the primary key and keep both Brnd_Name and Gnrc_Name.

In [ ]:
# Collect unique drug information from prescribers_df.
drug_dim = prescribers_df[["Brnd_Name", "Gnrc_Name", "Tot_Clms"]].copy()

# Normalize text so case/whitespace variants don't create false duplicates.
for col in ["Brnd_Name", "Gnrc_Name"]:
    drug_dim[col] = drug_dim[col].str.strip()

print("Missing:", drug_dim[["Brnd_Name", "Gnrc_Name"]].isna().sum().to_dict())
drug_dim = drug_dim.dropna(subset=["Brnd_Name", "Gnrc_Name"])

# Brand names that map to more than one generic name violate the PRIMARY KEY.
generics_per_brand = drug_dim.groupby("Brnd_Name")["Gnrc_Name"].nunique()
conflicts = generics_per_brand[generics_per_brand > 1].index
print("Brands:", generics_per_brand.shape[0], "| Brands with >1 generic:", len(conflicts))
drug_dim[drug_dim["Brnd_Name"].isin(conflicts)].groupby(["Brnd_Name", "Gnrc_Name"])["Tot_Clms"].sum().head(20)


In [ ]:
# Using a surrogate key drug_id for the drug dimension. since brand_name is not unique and cannot be used as a primary key
drug_df = (
    drug_dim[["Brnd_Name", "Gnrc_Name"]]
    .drop_duplicates()
    .rename(columns={"Brnd_Name": "brand_name", "Gnrc_Name": "generic_name"})
    .reset_index(drop=True)
)
drug_df.insert(0, "drug_id", range(1, len(drug_df) + 1))

assert drug_df["drug_id"].is_unique
assert drug_df["brand_name"].notna().all()
assert drug_df["generic_name"].notna().all()

print("Drug Row Count:", drug_df.shape[0])
drug_df.head()

##### PRESCRIPTION RESOLUTION

In [ ]:
prescription_df = (
    prescribers_df[["Prscrbr_NPI", "Tot_Clms", "Tot_Day_Suply", "Tot_Drug_Cst", "Tot_Benes", "Brnd_Name", "Gnrc_Name"]]
    .merge(drug_df, how="inner", left_on=["Brnd_Name", "Gnrc_Name"], right_on=["brand_name", "generic_name"])
    .reset_index(drop=True)[["Prscrbr_NPI", "drug_id", "Tot_Clms", "Tot_Day_Suply", "Tot_Drug_Cst", "Tot_Benes"]]
).rename(columns={"Prscrbr_NPI": "npi", "Tot_Clms": "total_claims", "Tot_Day_Suply": "total_day_supply", "Tot_Drug_Cst": "total_drug_cost", "Tot_Benes": "total_beneficiaries"})


# check that prescription has unique npi, drug_id combination
assert prescription_df.groupby(["npi", "drug_id"]).ngroups == prescription_df.shape[0]

print("Prescriptions Row Count: ", prescription_df.shape[0])
prescription_df.head()


In [ ]:
def npi_overlap(prescribers_df: pd.DataFrame, payments_df: pd.DataFrame, states=None) -> dict:
    """Distinct NPI overlap between prescribers_df and payments_df, optionally limited to prescriber states."""
    if isinstance(states, str):
        states = [states]

    pres = prescribers_df
    if states:
        pres = pres[pres["Prscrbr_State_Abrvtn"].isin(states)]

    pres_npis = set(pres["Prscrbr_NPI"].unique())
    pay_npis = set(payments_df["Covered_Recipient_NPI"].unique())
    overlap = pres_npis & pay_npis

    result = {
        "States": states or "ALL",
        "distinct_prescriber_npis": len(pres_npis),
        "distinct_overlap_npis": len(overlap),
        "overlap_pct": round(len(overlap) / len(pres_npis) * 100, 2) if pres_npis else 0.0,
    }
    return result

In [ ]:
print(npi_overlap(prescribers_df, payments_df, ["PA", "NJ", "DE"]))
print(npi_overlap(prescribers_df, payments_df, "NM"))
print(npi_overlap(prescribers_df, payments_df))

In [ ]:
states_of_interest = ["NM"]

# Filter doctors and prescriptions for states of interest
doctor = doctor_df[doctor_df["state"].isin(states_of_interest)]
prescription = prescription_df.merge(
    doctor_df[doctor_df["state"].isin(states_of_interest)][["npi"]],
    how="inner",
    on="npi",
)

# Filter payments for state of interest
raw_payments_filtered = payments_df.merge(
    doctor_df[doctor_df["state"].isin(states_of_interest)][["npi"]],
    how="inner",
    left_on="Covered_Recipient_NPI",
    right_on="npi",
).drop(columns="npi")

print("Presciption by state of interest: ", prescription.shape[0])
print("Payments by state of interest: ", raw_payments_filtered.shape[0])

In [59]:
# Analysis of prescribers and prescriptions for states of interest. Limiting to NPIs with payments reduces the dataset to relevant doctors
nm_rows = prescribers_df[
    prescribers_df["Prscrbr_State_Abrvtn"].isin(states_of_interest)
]
doctor_npis = doctor_df["npi"].dropna().unique()
nm_matches_doctor = nm_rows["Prscrbr_NPI"].isin(doctor_npis)

print("States of Interest prescriber rows:", len(nm_rows))
print("States of Interest distinct NPIs:", nm_rows["Prscrbr_NPI"].nunique())
print("States of Interest rows whose NPI is in doctor_df (Doctors with Payments):", nm_matches_doctor.sum())
print("Rows in prescription:", len(prescription))
print(
    "Distinct States of Interest NPIs in doctor_df:",
    nm_rows.loc[nm_matches_doctor, "Prscrbr_NPI"].nunique(),
)
print("Distinct States of Interest NPIs in prescription:", prescription["npi"].nunique())

States of Interest prescriber rows: 151371
States of Interest distinct NPIs: 6781
States of Interest rows whose NPI is in doctor_df (Doctors with Payments): 91302
Rows in prescription: 91302
Distinct States of Interest NPIs in doctor_df: 3240
Distinct States of Interest NPIs in prescription: 3240


##### PAYMENT RESOLUTION

In [60]:
payment_dim = raw_payments_filtered.copy()
payment_dim = (
    payment_dim[[
        "Record_ID",
        "Covered_Recipient_NPI",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID",
        "Total_Amount_of_Payment_USDollars",
        "Number_of_Payments_Included_in_Total_Amount",
        "Nature_of_Payment_or_Transfer_of_Value",
        "Form_of_Payment_or_Transfer_of_Value",
    ]]
    .rename(
        columns={
            "Record_ID": "record_id",
            "Covered_Recipient_NPI": "npi",
            "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID": "company_id",
            "Total_Amount_of_Payment_USDollars": "amount",
            "Number_of_Payments_Included_in_Total_Amount": "number_of_payments",
            "Nature_of_Payment_or_Transfer_of_Value": "nature_of_payments",
            "Form_of_Payment_or_Transfer_of_Value": "form_of_payments",
        }
    )
    .reset_index(drop=True)
)


##### PAYMENT BY DRUG RESOLUTION.

We'll spin off the 5 drugs associated with payments and use fuzzy matching to get their brand_names from the drugs table which was derived from prescription data.

We will only consider 80%+ for the scoring.

NOTE: Some payments are not for  drugs, but rather for medical devices which cannot be tied back to prescription table. We'll mostly use LEFT JOIN when linking payments to payments by drug.

In [61]:
from rapidfuzz import fuzz, process, utils
payment_name_columns = [
    f"Name_of_Drug_or_Biological_or_Device_or_Medical_Supply_{i}"
    for i in range(1, 6)
]

payment_name_values = (
    pd.concat(
        [raw_payments_filtered[col] for col in payment_name_columns],
        ignore_index=True,
    )
    .dropna()
    .astype("string")
    .str.strip()
)
payment_name_values = payment_name_values[
    payment_name_values.ne("")
].drop_duplicates()

payment_names = payment_name_values.to_frame(name="payment_name")
payment_names["_match_key"] = payment_names["payment_name"].str.casefold()

drug_name_lookup = drug_df[["drug_id", "brand_name", "generic_name"]].dropna(
    subset=["brand_name"]
).copy()
drug_name_lookup["_match_key"] = drug_name_lookup["brand_name"].str.strip().str.casefold()

resolved_payment_names = payment_names.merge(
    drug_name_lookup,
    on="_match_key",
    how="left",
).drop(columns="_match_key")
resolved_payment_names["drug_id"] = resolved_payment_names["drug_id"].astype("Int64")

matched_payment_names = resolved_payment_names[
    resolved_payment_names["drug_id"].notna()
]
unmatched_payment_names = resolved_payment_names[
    resolved_payment_names["drug_id"].isna()
]

print("Distinct names in filtered payments:", len(payment_names))
print("Distinct names matched exactly:", matched_payment_names["payment_name"].nunique())
print("Distinct names unmatched exactly:", unmatched_payment_names["payment_name"].nunique())

fuzzy_drug_lookup = drug_df[["drug_id", "brand_name", "generic_name"]].dropna(
    subset=["brand_name"]
).copy()
fuzzy_drug_lookup["_fuzzy_key"] = fuzzy_drug_lookup["brand_name"].map(
    utils.default_process
)
brand_choices = (
    fuzzy_drug_lookup[["_fuzzy_key", "brand_name"]]
    .dropna(subset=["_fuzzy_key"])
    .drop_duplicates(subset=["_fuzzy_key"])
    .set_index("_fuzzy_key")["brand_name"]
)
brand_keys = brand_choices.index.tolist()

def best_brand_candidate(payment_name):
    query = utils.default_process(payment_name)
    result = process.extractOne(query, brand_keys, scorer=fuzz.ratio)
    if result is None:
        return None, 0.0
    return result[0], result[1]

fuzzy_candidates = unmatched_payment_names[["payment_name"]].drop_duplicates().copy()
best_matches = [
    best_brand_candidate(name) for name in fuzzy_candidates["payment_name"]
]
fuzzy_candidates[["_fuzzy_key", "similarity_score"]] = pd.DataFrame(
    best_matches,
    index=fuzzy_candidates.index,
    columns=["_fuzzy_key", "similarity_score"],
)

fuzzy_name_suggestions = fuzzy_candidates.merge(
    fuzzy_drug_lookup.rename(columns={"brand_name": "candidate_brand_name"}),
    on="_fuzzy_key",
    how="left",
)
fuzzy_name_suggestions["drug_id"] = fuzzy_name_suggestions["drug_id"].astype("Int64")
fuzzy_name_suggestions["similarity_score"] = fuzzy_name_suggestions[
    "similarity_score"
].round(1)

scores = fuzzy_candidates["similarity_score"]
print("Unmatched names with score >= 90:", int(scores.ge(90).sum()))
print("Unmatched names with score 80-89:", int(scores.ge(80).sum() - scores.ge(90).sum()))
print("Unmatched names with score < 80:", int(scores.lt(80).sum()))

fuzzy_name_suggestions[
    fuzzy_name_suggestions["similarity_score"].ge(80)
].sort_values("similarity_score", ascending=False).head(25)

Distinct names in filtered payments: 1555
Distinct names matched exactly: 433
Distinct names unmatched exactly: 1122
Unmatched names with score >= 90: 1
Unmatched names with score 80-89: 42
Unmatched names with score < 80: 1079


,payment_name,_fuzzy_key,similarity_score,drug_id,candidate_brand_name,generic_name
884,NUPRO,neupro,90.9,549,Neupro,Rotigotine
161,GARDASIL,gardasil 9,88.9,1154,Gardasil 9,Hpv Vaccine 9-Valent/Pf
279,CORE,coreg,88.9,2453,Coreg,Carvedilol
285,NP Thyroid 60,np thyroid,87.0,983,Np Thyroid,"Thyroid,pork"
198,RENASYS TOUCH,easy touch,87.0,2475,Easy Touch,"Syringe And Needle,insulin,1ml"
197,RENASYS TOUCH,easy touch,87.0,1949,Easy Touch,"Syring-Needl,disp,insul,0.3 Ml"
196,RENASYS TOUCH,easy touch,87.0,1865,Easy Touch,"Syringe-Needle,insulin,0.5 Ml"
370,ORENITRAM,orenitram er,85.7,1712,Orenitram Er,Treprostinil Diolamine
993,Avance,dalvance,85.7,1184,Dalvance,Dalbavancin Hcl
327,TWINFIX,twinrix,85.7,1160,Twinrix,Hepatitis A And B Vaccine/Pf


In [62]:
# Keep exact matches and fuzzy suggestions with similarity of at least 80.
exact_drug_matches = matched_payment_names[
    ["payment_name", "drug_id", "brand_name", "generic_name"]
].copy()
exact_drug_matches["match_type"] = "exact"
exact_drug_matches["similarity_score"] = 100.0

fuzzy_drug_matches = fuzzy_name_suggestions.loc[
    fuzzy_name_suggestions["similarity_score"].ge(80),
    [
        "payment_name",
        "drug_id",
        "candidate_brand_name",
        "generic_name",
        "similarity_score",
    ],
].rename(
    columns={"candidate_brand_name": "brand_name"}
).copy()
fuzzy_drug_matches["match_type"] = "fuzzy"

resolved_drugs_list = pd.concat(
    [exact_drug_matches, fuzzy_drug_matches],
    ignore_index=True,
)
resolved_drugs_list["drug_id"] = resolved_drugs_list["drug_id"].astype("Int64")
resolved_drugs_list = resolved_drugs_list[
    ["payment_name", "drug_id", "brand_name", "generic_name", "match_type", "similarity_score"]
]

print("Exact matches:", exact_drug_matches["payment_name"].nunique())
print("Fuzzy matches with score >= 80:", fuzzy_drug_matches["payment_name"].nunique())
resolved_drugs_list.head()

Exact matches: 433
Fuzzy matches with score >= 80: 43


,payment_name,drug_id,brand_name,generic_name,match_type,similarity_score
0,OPSUMIT,921,Opsumit,Macitentan,exact,100.0
1,INVEGA SUSTENNA,495,Invega Sustenna,Paliperidone Palmitate,exact,100.0
2,STELARA,725,Stelara,Ustekinumab,exact,100.0
3,TREMFYA,1086,Tremfya,Guselkumab,exact,100.0
4,RYBREVANT,2799,Rybrevant,Amivantamab-Vmjw,exact,100.0


In [63]:
# Convert the five product-name slots to one row per nonblank payment/product.
payment_name_columns = [
    f"Name_of_Drug_or_Biological_or_Device_or_Medical_Supply_{i}"
    for i in range(1, 6)
]
payment_id_vars = [
    column for column in raw_payments_filtered.columns
    if column not in payment_name_columns
]
payment_name_rows = raw_payments_filtered.melt(
    id_vars=payment_id_vars,
    value_vars=payment_name_columns,
    var_name="drug_name_column",
    value_name="payment_name",
)
payment_name_rows["payment_name"] = (
    payment_name_rows["payment_name"].astype("string").str.strip()
)
payment_name_rows = payment_name_rows.dropna(subset=["payment_name"])
payment_name_rows = payment_name_rows[
    payment_name_rows["payment_name"].ne("")
].copy()
payment_name_rows["_payment_name_key"] = payment_name_rows[
    "payment_name"
].str.casefold()

resolved_drug_matches = resolved_drugs_list.copy()
resolved_drug_matches["_payment_name_key"] = resolved_drug_matches[
    "payment_name"
].str.strip().str.casefold()

payment_by_drug_init = payment_name_rows.merge(
    resolved_drug_matches[
        [
            "_payment_name_key",
            "drug_id",
            "brand_name",
            "generic_name",
            "match_type",
            "similarity_score",
        ]
    ],
    on="_payment_name_key",
    how="inner",
).drop(columns="_payment_name_key")
payment_by_drug_init["drug_id"] = payment_by_drug_init["drug_id"].astype("Int64")

print("Matched payment/product rows:", len(payment_by_drug_init))
payment_by_drug_init.head()

Matched payment/product rows: 31715


,Change_Type,Covered_Recipient_Type,Teaching_Hospital_CCN,Teaching_Hospital_ID,Teaching_Hospital_Name,Covered_Recipient_Profile_ID,Covered_Recipient_NPI,Covered_Recipient_First_Name,Covered_Recipient_Middle_Name,Covered_Recipient_Last_Name,...,Associated_Device_or_Medical_Supply_PDI_5,Program_Year,Payment_Publication_Date,drug_name_column,payment_name,drug_id,brand_name,generic_name,match_type,similarity_score
0,UNCHANGED,Covered Recipient Non-Physician Practitioner,NaN,NaN,NaN,10543572.0,1912050089,CHARLES,JOHN,GALLEGOS,...,NaN,2024,06/30/2026,Name_of_Drug_or_Biological_or_Device_or_Medica...,OPSUMIT,921,Opsumit,Macitentan,exact,100.0
1,UNCHANGED,Covered Recipient Non-Physician Practitioner,NaN,NaN,NaN,11357128.0,1417398298,RANJANA,NaN,VERMA,...,NaN,2024,06/30/2026,Name_of_Drug_or_Biological_or_Device_or_Medica...,INVEGA SUSTENNA,495,Invega Sustenna,Paliperidone Palmitate,exact,100.0
2,UNCHANGED,Covered Recipient Non-Physician Practitioner,NaN,NaN,NaN,10511542.0,1437485893,SVETLANA,K,BROTHERS,...,NaN,2024,06/30/2026,Name_of_Drug_or_Biological_or_Device_or_Medica...,STELARA,725,Stelara,Ustekinumab,exact,100.0
3,UNCHANGED,Covered Recipient Non-Physician Practitioner,NaN,NaN,NaN,10527635.0,1699292920,ALLISON,K,MILLS,...,NaN,2024,06/30/2026,Name_of_Drug_or_Biological_or_Device_or_Medica...,STELARA,725,Stelara,Ustekinumab,exact,100.0
4,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,913443.0,1326337122,MICHAEL,B,CHANG,...,NaN,2024,06/30/2026,Name_of_Drug_or_Biological_or_Device_or_Medica...,TREMFYA,1086,Tremfya,Guselkumab,exact,100.0


In [64]:
payment_by_drug = (
    payment_by_drug_init[["Record_ID", "drug_id"]]
).rename(columns={"Record_ID": "record_id"}).reset_index(drop=True)

##### CSV OUTPUT

In [65]:
# Export to csv if ok with the chosen states of interest
company_df.to_csv(OUTPUT_DIR / CLEANED_SET["company"], index=False)
doctor.to_csv(OUTPUT_DIR / CLEANED_SET["doctor"], index=False)
drug_df.to_csv(OUTPUT_DIR / CLEANED_SET["drug"], index=False)
prescription.to_csv(OUTPUT_DIR / CLEANED_SET["prescription"], index=False)
payment_dim.to_csv(OUTPUT_DIR / CLEANED_SET["payment"], index=False)
payment_by_drug.to_csv(OUTPUT_DIR / CLEANED_SET["payment_by_drug"], index=False)

# Raw payments filtered by state of interest
# raw_payments_filtered.to_csv(OUTPUT_DIR / "raw_payments_filtered.csv", index=False)

NameError: name 'company_df' is not defined